In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/dylanmalis-collab/Exoplanet_Life/refs/heads/elt_v2/Exoplanet_DBT/data_exports/raw_exoplanets_enriched.csv"
df = pd.read_csv(url)

C:\Users\berna\AppData\Local\Temp\ipykernel_3688\287856858.py:3: DtypeWarning: Columns (0: hd_name, 1: pl_orbtperstr, 2: pl_orbtper_reflink, 3: pl_orbtper_systemref, 4: pl_trueobliqstr, 5: pl_trueobliq_reflink, 6: st_log_rhkstr, 7: st_log_rhk_reflink, 8: sy_icmagstr, 9: pl_cmassjstr, 10: pl_cmassj_reflink, 11: pl_cmassestr, 12: pl_cmasse_reflink) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


In [2]:
#première selection grossière
df = df[["pl_name", "hostname", "pl_orbsmax_calc","pl_rade", "pl_bmasse_calc", "pl_dens_calc", "pl_orbeccen", "pl_insol_calc", "pl_eqt_calc","st_spectype_group_en", "st_teff", "st_rad", "st_mass", "st_met","st_lum", "st_lum_lin_calc", "st_dens", "Gravity_G_earth_calc", "masse_10_24kg_calc","pl_type", "sim_earth", "eau_possible","pt_habitable"]]

In [3]:
#transformation (Nan et création des colonne missing)
df["pl_orbeccen"] = df["pl_orbeccen"].fillna(df["pl_orbeccen"].median())

df["st_teff_missing"] = df["st_teff"].isna().astype(int)

# 1. imputation par groupe
df["st_teff"] = df.groupby("st_spectype_group_en")["st_teff"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_teff"] = df["st_teff"].fillna(df["st_teff"].median())

df["st_mass_missing"] = df["st_mass"].isna().astype(int)

# 1. imputation par groupe
df["st_mass"] = df.groupby("st_spectype_group_en")["st_mass"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_mass"] = df["st_mass"].fillna(df["st_mass"].median())

df["st_rad_missing"] = df["st_rad"].isna().astype(int)

# 1. imputation par groupe
df["st_rad"] = df.groupby("st_spectype_group_en")["st_rad"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_rad"] = df["st_rad"].fillna(df["st_rad"].median())

df["st_met_missing"] = df["st_met"].isna().astype(int)

# 1. imputation par groupe
df["st_met"] = df.groupby("st_spectype_group_en")["st_met"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_met"] = df["st_met"].fillna(df["st_met"].median())

df["st_dens_missing"] = df["st_dens"].isna().astype(int)

# 1. imputation par groupe
df["st_dens"] = df.groupby("st_spectype_group_en")["st_dens"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_dens"] = df["st_dens"].fillna(df["st_dens"].median())

df["st_lum_calc"] = df["st_lum"].fillna(df.groupby("st_spectype_group_en")["st_lum"].transform("median"))

In [4]:
# juste pl_insol
df4 = df[["pl_orbsmax_calc","pl_rade", "pl_bmasse_calc", "pl_dens_calc", "pl_orbeccen", "st_teff", "st_rad", "st_mass", "st_met", "st_lum_calc", "st_dens", "sim_earth","pl_insol_calc"]]

In [5]:
from sklearn.model_selection import train_test_split

X = df4
y = df["pt_habitable"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y, random_state=42)

In [6]:
import sys
print(sys.executable)

j:\Projet 3\.venv\Scripts\python.exe


In [6]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
#from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
cv = RepeatedStratifiedKFold( n_splits=5, n_repeats=3, random_state=42 )

In [7]:
XGB = XGBClassifier(random_state=42,tree_method="hist")

XGB_param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.5, 0.8],
    "gamma": [0, 0.1],
    "min_child_weight": [1, 5, 10],
    "scale_pos_weight": [10, 30, 50],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [1, 5],
    "max_delta_step": [0, 1]
}

XGB_grid = GridSearchCV(
    XGB,
    XGB_param_grid,
    cv=cv,
    scoring="average_precision",
    refit=True,
    n_jobs=-1
)

XGB_grid.fit(X_train, y_train)

print("\nXGBOOST")
print("Best average_precision:", XGB_grid.best_score_)
print("Best params:", XGB_grid.best_params_)


XGBOOST
Best average_precision: 0.8547089947089946
Best params: {'colsample_bytree': 0.5, 'gamma': 0, 'learning_rate': 0.05, 'max_delta_step': 1, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 300, 'reg_alpha': 0.1, 'reg_lambda': 5, 'scale_pos_weight': 50, 'subsample': 1.0}


In [8]:
model_xgb = XGB_grid.best_estimator_

In [9]:
from sklearn.metrics import average_precision_score

y_proba = model_xgb.predict_proba(X_test)[:, 1]

ap = average_precision_score(y_test, y_proba)

print(" Average Precision:", ap)

 Average Precision: 1.0


In [10]:
from sklearn.metrics import f1_score

y_pred = model_xgb.predict(X_test)

f1 = f1_score(y_test, y_pred)

print("F1 score (threshold=0.5):", f1)

F1 score (threshold=0.5): 0.6666666666666666


In [11]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1230
           1       1.00      0.50      0.67         2

    accuracy                           1.00      1232
   macro avg       1.00      0.75      0.83      1232
weighted avg       1.00      1.00      1.00      1232



In [12]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# calcul F1 (avec protection division par zéro)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

# IMPORTANT : on aligne avec thresholds
f1_scores = f1_scores[:-1]   # on enlève le dernier point

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

y_pred2 = (y_proba >= best_threshold).astype(int)

print("F1 final:", f1_score(y_test, y_pred2))

Best threshold: 0.24392846
F1 final: 1.0


In [16]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_proba_train = model_xgb.predict_proba(X_train)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, y_proba_train)

f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)
f1_scores = f1_scores[:-1]

best_idx = np.argmax(f1_scores)
best_threshold2 = thresholds[best_idx]

print("Best threshold (train):", best_threshold2)

Best threshold (train): 0.99369


In [17]:
from sklearn.metrics import f1_score

y_proba_test = model_xgb.predict_proba(X_test)[:, 1]

y_pred_test = (y_proba_test >= best_threshold).astype(int)

print("F1 test:", f1_score(y_test, y_pred_test))

F1 test: 1.0


In [13]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model_xgb,
    X_train,
    y_train,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1
)


In [14]:
print("\n📈 CV stability (Average Precision)")
print("Mean:", cv_scores.mean())
print("Std :", cv_scores.std())


📈 CV stability (Average Precision)
Mean: 0.8547089947089946
Std : 0.1831674199820927


In [15]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

threshold = 0.1466639311252936  # ton seuil optimisé

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in cv.split(X_train, y_train):

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = clone(model_xgb)
    model.fit(X_tr, y_tr)

    y_proba = model.predict_proba(X_val)[:, 1]
    y_pred3 = (y_proba >= threshold).astype(int)

    f1 = f1_score(y_val, y_pred3)
    f1_scores.append(f1)

print("F1 CV mean:", np.mean(f1_scores))
print("F1 CV std :", np.std(f1_scores))

F1 CV mean: 0.72
F1 CV std : 0.2712931993250107
